# Mach-Zehnder Interferometer
A Mach-Zehnder interferometer is a two-arm interferometric setup used to compare the phase of two coherent optical waves. A laser beam is first divided by a beam splitter into an object arm and a reference arm. The object beam interacts with the sample, accumulating a phase shift determined by the optical path length, while the reference beam propagates through a separate, usually undisturbed path. The two beams are then recombined at a second beam splitter and made to interfere at the detector. Since the camera measures intensity rather than phase, the recorded signal is

$$ I(x,y) = |E_o + E_r|^2 = |E_o|^2 + |E_r|^2 + E_oE_r^* + E_o^*E_r $$

The first two terms are the individual intensities of the object and reference beams. The last two terms are interference terms, which depend on the relative phase between the beams. For scalar fields, this can be written as

$$ I(x,y) = I_o + I_r + 2\sqrt{I_o I_r} cos[\Delta\phi(x,y)] $$

The interferometer as a configuration as follows


<p align="center">
<img src="img/Mach-Zehnder.png" width="600"/>
</p>


---
## 1. Aligning the interferometer
Open the camera and display the live image from the detector. Make sure that both beams from the interferometer are reaching the camera and overlapping on the sensor. You should observe a pattern of bright and dark fringes, which comes from the interference between the object beam and the reference beam. Adjust one of the mirrors, slightly changing the angle of one beam, and observe how the fringes move, rotate, or change spacing. This shows how small changes in optical path length or beam alignment modify the phase difference between the two beams, which is then converted into a visible intensity pattern on the camera.

In [ ]:
# Required dependencies
# !pip install opencv-python numpy matplotlib scikit-image pillow

In [ ]:
from MZ_camera import *
run_camera_gui()

---
## 2. Off-axis digital holography: Phase imaging of a transparent sample

In off-axis digital holography, the Mach-Zehnder interferometer is adjusted so that the object and reference beams do not recombine perfectly parallel to each other. Instead, the reference beam reaches the camera with a small angle relative to the object beam. This angle produces a regular fringe pattern across the detector, called a *spatial carrier*.

<p align="center">
<img src="img/OffAxisHolography.png" width="600"/>
</p>



 At the detector plane, the reference field can be approximated as

$$
E_r(x,y)=A_r e^{i(k_xx+k_yy)},
$$

where $k_x$ and $k_y$ describe the tilt of the reference beam in the camera plane. If the object field is written as

$$
E_o(x,y)=A_o(x,y)e^{i\phi_o(x,y)},
$$

then the intensity measured by the camera is

$$
I(x,y)=|E_o+E_r|^2.
$$

Assuming a uniform reference amplitude, this can be written as

$$
I(x,y)=I_o+I_r+2\sqrt{I_oI_r}
\cos[\phi_o(x,y)-k_xx-k_yy].
$$

The best way to observe the separation of the different contributions is to calculate the Fourier transform of the off-axis hologram. For a reference beam tilted along the $y$ direction, the Fourier transform can be written schematically as

$$
\begin{aligned}
\mathcal{F}\{I(x,y)\}
=&\;
\underbrace{|E_r|^2 \delta(f_x,f_y)}_{\text{zero-order DC}}
+
\underbrace{
\mathcal{F}\{|E_o|^2\}(f_x,f_y)
}_{\text{zero-order object intensity}}
\\[6pt]
&+
\underbrace{
E_r^* \mathcal{F}\{E_o\}
\left(f_x, f_y-\frac{\sin\theta}{\lambda}\right)
}_{\text{+1st order}}
+
\underbrace{
E_r \mathcal{F}\{E_o^*\}
\left(f_x, f_y+\frac{\sin\theta}{\lambda}\right)
}_{\text{-1st order}} .
\end{aligned}
$$

The zero-order terms are centered around the spatial frequency $0$. These terms contain the reference intensity and the object intensity, but they do not directly give the phase of the object field. The $\pm 1$ diffraction orders are shifted away from the origin by the spatial carrier frequency

$$
f_c=\frac{\sin\theta}{\lambda}=\frac{k\sin\theta}{2\pi}.
$$

This separation is the key advantage of the off-axis configuration. Since the reference field is known and approximately constant, one of the first-order terms is proportional to the complex object field $E_o$, while the other is proportional to its complex conjugate $E_o^*$. Therefore, we can numerically select one sideband in the Fourier domain, shift it back to the origin, and apply an inverse Fourier transform. This gives access to the complex object field.

Visually, the process is:

<p align="center">
<img src="img/fourier_domain.png" width="400"/>
</p>

After filtering and inverse Fourier transforming the selected first-order term, the reconstructed field is

$$
E_o(x,y)=A_o(x,y)e^{i\phi_o(x,y)}.
$$

From this complex field, the amplitude and phase are obtained as

$$
A_o(x,y)=|E_o(x,y)|,
$$

and

$$
\phi_o(x,y)=\arg[E_o(x,y)].
$$


### Activity
1. Block reference path to record the profile.
2. Unblock reference path and record the interferogram profile.
3. Compute the Fourier spectrum (use log-scale).
4. Filter the Fourier transform around -1 order.
5. Roll the FFT to the center, by applying (kx,ky) translation.
6. Apply the inverse FFT and plot the recovered phase profile.

In [ ]:
## Helper functions -> acquires image from camera
import cv2
def acquire_image(
    camera_index=1,
    filename="acquired_image.png",
    grayscale=False,
    show_image=False,
):
    """
    Acquire one image from a camera and save it.

    Parameters
    ----------
    camera_index : int
        Camera index. Usually 0 for the default camera.
    filename : str
        Name of the output image file.
    grayscale : bool
        If True, save the image in grayscale.
    show_image : bool
        If True, display the acquired image.

    Returns
    -------
    image : numpy.ndarray
        The acquired image.
    """

    camera = cv2.VideoCapture(camera_index)

    if not camera.isOpened():
        raise RuntimeError(f"Could not open camera {camera_index}.")

    ret, frame = camera.read()

    camera.release()

    if not ret:
        raise RuntimeError("Could not acquire image.")

    if grayscale:
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        image = frame

    cv2.imwrite(filename, image)

    if show_image:
        cv2.imshow("Acquired Image", image)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

    print(f"Image saved as {filename}")

    return image

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
%matplotlib qt

#dynamic display of the plots: use %matplotlib qt, %matplotlib tk or %matplotlib widget
#from matplotlib.widgets import Slider

#1. Load reference and hologram images (use imread as_gray = True)
interferogram = imread('hologram_group1.png',as_gray=True).astype(np.float32)
profile = imread('reference_group1.png',as_gray=True).astype(np.float32)

#2. Compute the Fourier transform of the hologram and shift zero-frequency to center (fftshift), plot its magnitude. For visualization use a logarithmic scale.
FT_hologram = np.fft.fftshift(np.fft.fft2(interferogram))
FT_mag = np.abs(FT_hologram)
FT_log = np.log(1 + FT_mag)

# Dimensions of the image.
nx, ny = interferogram.shape

# Create spatial grids (used later for phase calculations if needed).
Y, X = np.indices((nx, ny))

fig, ax = plt.subplots(1, 3, figsize=(10, 4))

# Middle: Display the interference image.
im1 = ax[0].imshow(profile, cmap='gray')
ax[0].set_title("Profile Image (no interference)")
ax[0].axis('off')

# Middle: Display the interference image.
im1 = ax[1].imshow(interferogram, cmap='gray')
ax[1].set_title("Interference Image")
ax[1].axis('off')

# Right: Display the log Fourier Transform of the interferogram
im2 = ax[2].imshow(FT_log, cmap='viridis')
ax[2].set_title("Fourier Transform (log scale)")
ax[2].axis('off')
fig.tight_layout()
plt.show()


In [ ]:
#4. Filter + Roll
# Set Up Filter Parameters (radius + center of the -1 order)
radius = 20
current_center = [623,330]  # [x, y] in FT (shifted) coordinates

# Apply a circular mask around the center of the -1 order
Y_ft, X_ft = np.indices((nx, ny))
mask = ((X_ft - current_center[0])**2 + (Y_ft - current_center[1])**2) <= radius**2
FT_filtered = FT_hologram * mask

# Compute the integer shifts needed to roll the selected center to the Fourier center.
shift_x = int(np.round((ny / 2) - current_center[0]))
shift_y = int(np.round((nx / 2) - current_center[1]))

# Roll the Fourier data: this centers the off-axis component, removing the phase tilt (hint: use np.roll)
FT_filtered_rolled = np.roll(FT_filtered, shift=(shift_y, shift_x), axis=(0, 1))


#5. Inverse Fourier transform to reconstruct the complex field.
# Unshift the filtered and rolled Fourier transform
#Calculate its inverse transform
#The reconstructed phase will be given by the angle of the inverse transform
E_rec = np.fft.ifft2(np.fft.ifftshift(FT_filtered_rolled))
reconstructed_phase = np.angle(E_rec)

#6. Visualize. Plot reference image and the reconstructed phase

fig, ax = plt.subplots(1, 3, figsize=(10, 4))

# Middle: Display the interference image.
im1 = ax[0].imshow(profile, cmap='gray')
ax[0].set_title("Profile Image (no interference)")
ax[0].axis('off')

# Middle: Display the filtered interference in Fourier space.
im1 = ax[1].imshow(np.log10(1+np.abs(FT_filtered)), cmap='viridis')
ax[1].set_title("Filtered interferogram")
ax[1].axis('off')

# Right: Display the log Fourier Transform of the interferogram
im2 = ax[2].imshow(reconstructed_phase, cmap='twilight', interpolation='nearest')
ax[2].set_title(r"$\phi(x,y)$")
ax[2].axis('off')
fig.tight_layout()




---
---
# Active homodyne detection using a Michelson Interferometer
<p align="center">
<img src="img/Michelson.png" width="400"/>
</p>


A Michelson interferometer maps optical path length differences onto intensity through coherent interference. For a quasi-monochromatic field, the detected power can be expressed as

$$I = \frac{I_0}{2} (1 + \cos{2\phi}) = \frac{I_0}{2} (1 + \cos{\frac{4 \pi \Delta L}{\lambda}})$$

where $\lambda$ is the light wavelength, $I_0$ is the intensity of the input light, and $\Delta L$ is the path difference between the reference and the measurement arm. This means that, if the length of the measurement arm changes, the interferences pattern will also shift accordingly. The next images illustrate this effect.

<p align="center">
<img src="img/interference.gif" width="500"/>
</p>

The plot on the right illustrates the variation on the intensity at the center of the interference pattern, similar to what would happen if you place a photodetector at that position. 

---
## Activitiy 1. Align and listen
In this activity, the Michelson interferometer acts as a displacement-to-phase-to-voltage transducer. Any perturbation that changes the optical path difference between the two arms, such as mirror motion, acoustic vibration, air currents, or mechanical drift, changes the relative phase of the recombined beams. This phase variation modulates the optical power reaching the photodetector.

The photodetector converts these optical power fluctuations into an electrical voltage. By sending this voltage to a speaker, the interferometer output becomes audible. The sound is therefore not the optical field itself, but the electrical representation of phase-induced intensity fluctuations.

This is closely related to the detection principle used in gravitational-wave interferometers such as LIGO. In both cases, a differential arm-length change produces a phase shift, which is converted into an optical power variation at the interferometer output and then into an electrical signal by a photodetector. The tabletop setup responds to macroscopic vibrations and acoustic perturbations, while LIGO is engineered to detect extremely small differential length changes produced by gravitational-wave strain.

The experimental setup is the following:
<p align="center">
<img src="img/michelson_act1.png" width="500"/>
</p>


Observe:
- Mechanical perturbations produce audible changes. 
- The response depends on the operating point of the interferometer.


---
## Activity 2 — Active homodyne detection

A key limitation of a free-running interferometer is that its operating point is not stable. Small environmental perturbations, such as temperature fluctuations, air currents, acoustic noise, or mechanical vibrations, slowly change the relative phase between the two interferometer arms. As a result, the detected intensity can drift across bright and dark fringes. In some regions of the fringe, the response to small phase variations is strong and approximately linear; in others, the response is weak or nearly zero.

The goal of active homodyne detection is to keep the interferometer close to quadrature, the operating point halfway between a maximum and a minimum of the interference fringe. At quadrature, small phase changes produce the largest approximately linear change in detected intensity. This makes the interferometer suitable for sensitive detection of small path-length or displacement variations.


<p align="center">
<img src="img/quadrature.png" width="600"/>
</p>


The experimental setup is the following:
<p align="center">
<img src="img/michelson_act2.png" width="500"/>
</p>

Activity:
Write a feedback algorithm that continuously:

1. reads the photodetector voltage from the DAQ,
2. compares it with a chosen target value corresponding to quadrature,
3. computes an error signal,
4. updates the voltage applied to the piezo actuator,
5. repeats this process in real time.

By adjusting the feedback parameters, you should observe how the system changes from a freely drifting interferometer to a stabilized phase-sensitive detector. The main objective is not only to lock the interferometer, but also to understand how feedback gain, update rate, noise, and actuator limits affect the stability of the lock.

In [ ]:
#Required dependencies
# !pip install nidaqmx tqdm

### With no feedback loop

First observe the output signal with no feedback loop. Observe the noise due to the external perturbations.

In [ ]:
#Import libraries
import time
import numpy as np
import matplotlib.pyplot as plt
import nidaqmx
from nidaqmx.constants import TerminalConfiguration
from tqdm import tqdm

#Configure parameters
device = "Dev5"
ao_channel = f"{device}/ao0"
ai_channel = f"{device}/ai0"


def read_photodetector(ai_channel, ao_channel, v_min, v_max = 10, n_pts = 100, num_reads = 1, delay = 0.005, plot = False):
    """Create a sweep of voltages for the piezoelectric actuator and reads photodetector voltage.

    Args:
        ai_channel (_type_): Input channel for photodetector voltage reading.
        ao_channel (_type_): Output channel for piezoelectric actuator voltage.
        v_min (_type_): Minimum voltage for the sweep.
        v_max (int, optional): Maximum voltage for the sweep. Defaults to 10. Max is 10V for the AO channel.
        n_pts (int, optional): Number of points in the sweep. Defaults to 100.
        num_reads (int, optional): Number of reads at each voltage point. Defaults to 1.
        delay (float, optional): Delay between reads. Defaults to 0.005.
        plot (bool, optional): Whether to plot the results. Defaults to False.

    Returns:
        ao_values (_type_): The sweep of voltages for the piezoelectric actuator.
        all_ai_values (_type_): The measured photodetector voltages.
    """

    ao_values = np.linspace(v_min, v_max, n_pts)
    all_ai_values = np.zeros((num_reads, n_pts))

    with nidaqmx.Task() as ao_task, nidaqmx.Task() as ai_task:
        ao_task.ao_channels.add_ao_voltage_chan(ao_channel)
        ai_task.ai_channels.add_ai_voltage_chan(
            ai_channel,
            terminal_config=TerminalConfiguration.RSE
        )


        for i, v in enumerate(tqdm(ao_values, desc="Measuring", unit="step")):
            ao_task.write(float(v))

            for j in range(num_reads):
                y = ai_task.read()
                all_ai_values[j, i] = y

    if plot:
        fig, ax = plt.subplots(1, 2, figsize=(10,4))

        for j in range(num_reads):
            ax[0].plot(ao_values, all_ai_values[j], label=f"Read {j+1}")

        ax[0].set_xlabel("Input Voltage (V)")
        ax[0].set_ylabel("Photodetector Measured Voltage (V)")
        ax[0].grid(True)
        ax[1].plot(ao_values, np.mean(all_ai_values, axis=0), label="Mean")
        ax[1].set_xlabel("Input Voltage (V)")
        ax[1].set_ylabel("Photodetector Measured Voltage (V)")
        ax[1].set_title("Mean Photodetector Readings")
        ax[1].grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

    return ao_values, all_ai_values


ao_values, all_ai_values = read_photodetector(ai_channel, ao_channel, v_min=0, v_max=10, n_pts=100, num_reads=5, delay=0.005, plot=True)

### Active homodyne feedback loop
1. **Set the feedback target**  
   The target photodetector voltage is chosen to correspond approximately to quadrature, where the interferometer has a strong and nearly linear phase-to-intensity response.

2. **Read the photodetector signal**  
   At each iteration, the code acquires the current photodetector voltage from the DAQ.

3. **Compute the error signal**  
   The measured voltage is compared with the target value. The difference between them is the error signal.

4. **Calculate the feedback correction**  
   The feedback correction can be calculated using proportional, PI, or PID control. In proportional control, the correction depends only on the current error. In PI control, an integral term is added to compensate slow drift. In PID control, a derivative term can also be included to react to rapid changes in the error.

   For this experiment, start with proportional control:

   *correction = kp * error*

   If the interferometer still drifts slowly, add an integral term:

   *correction = kp * error + ki * integral_error*

   A full PID correction has the form:
   ```python
   correction = (
    kp * error
    + ki * integral_error
    + kd * derivative_error)

5. **Update the piezo voltage**  
   The correction is added to the previous piezo voltage. This changes the optical path difference in one interferometer arm.

6. **Apply voltage limits**  
   The piezo voltage is clipped between the minimum and maximum allowed values to protect the actuator and DAQ output.

7. **Store the data**  
   Time, photodetector voltage, error, and piezo voltage are saved during the loop.

8. **Plot the results**  
   After the loop, the recorded data are plotted to evaluate whether the feedback stabilized the interferometer.

The objective is to adjust the feedback parameters, especially the proportional gain and optionally the integral gain, until the photodetector signal remains close to the target value without oscillating or saturating the piezo voltage.

In [ ]:
# ============================================================
# Active homodyne feedback loop
# ============================================================


# voltage limits
v_min = 0.0               # minimum piezo voltage [V]
v_max = 10.0              # maximum piezo voltage [V], maximum allowed by the DAQ is 10 V
piezo_voltage = 5.0     # initial piezo voltage [V]

# feedback parameters
target_pd = 2.5           # target photodetector voltage [V] -> look at the previous result and choose a value near the quadrature point

kp = 0.02                 # proportional gain
ki = 0.00                 # integral gain; start with zero
kd = 0.00                 # derivative gain; start with zero

dt = 0.02                 # feedback loop time step [s]
duration = 20.0           # total acquisition time [s]

n_samples = int(duration / dt)

#Result arrays
time_array = np.zeros(n_samples)
pd_array = np.zeros(n_samples)
error_array = np.zeros(n_samples)
piezo_array = np.zeros(n_samples)
correction_array = np.zeros(n_samples)

# Feedback loop
def read_photodetector(ai_task):
    """
    Read the photodetector voltage from the DAQ.
    """
    return ai_task.read()


def write_piezo(ao_task, piezo_voltage, v_min=0.0, v_max=10.0):
    """
    Write voltage to the piezo actuator with safety clipping.
    """
    piezo_voltage = np.clip(piezo_voltage, v_min, v_max)
    ao_task.write(piezo_voltage)
    return piezo_voltage


with nidaqmx.Task() as ai_task, nidaqmx.Task() as ao_task:

    ai_task.ai_channels.add_ai_voltage_chan(
        ai_channel,
        terminal_config=TerminalConfiguration.RSE,
        min_val=-10.0,
        max_val=10.0
    )

    ao_task.ao_channels.add_ao_voltage_chan(
        ao_channel,
        min_val=v_min,
        max_val=v_max
    )

    write_piezo(ao_task, piezo_voltage, v_min, v_max)

    t0 = time.time()

    for i in range(n_samples):

        loop_start = time.time()

        # Read photodetector voltage
        pd_voltage = read_photodetector(ai_task)

        # PID feedback
        error = target_pd - pd_voltage

        integral_error = integral_error + error * dt

        derivative_error = (error - previous_error) / dt

        correction = (
            kp * error
            + ki * integral_error
            + kd * derivative_error
        )

        piezo_voltage = piezo_voltage + correction

        piezo_voltage = write_piezo(
            ao_task,
            piezo_voltage,
            v_min,
            v_max
        )

        previous_error = error

        # Store data
        time_array[i] = time.time() - t0
        pd_array[i] = pd_voltage
        piezo_array[i] = piezo_voltage
        error_array[i] = error
        correction_array[i] = correction

        # Keep loop timing approximately constant
        elapsed = time.time() - loop_start
        if elapsed < dt:
            time.sleep(dt - elapsed)



plt.figure(figsize=(8, 4))
plt.plot(time_array, pd_array, label="Photodetector signal")
plt.axhline(target_pd, linestyle="--", label="Target")
plt.xlabel("Time [s]")
plt.ylabel("Photodetector voltage [V]")
plt.title("Photodetector signal")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(time_array, piezo_array)
plt.xlabel("Time [s]")
plt.ylabel("Piezo voltage [V]")
plt.title("Piezo control voltage")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(time_array, error_array)
plt.xlabel("Time [s]")
plt.ylabel("Error [V]")
plt.title("Feedback error")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(time_array, correction_array)
plt.xlabel("Time [s]")
plt.ylabel("Correction [V]")
plt.title("PID correction")
plt.grid(True)
plt.show()